# LIMPIEZA DE DATOS — PIB (CNT)
### Contabilidad Nacional Trimestral de España: Base 1986 (1970–1998) + INE (1995–2026)
### Incluye:
- Importación de librerías
- Carga de datasets
- Conversión de tipos y limpieza
- Conversión de pesetas a euros (tasa irrevocable 1 EUR = 166,386 pts)
- Homogenización MECE de categorías disponibles 1970–2026
- Verificación del solapamiento 1995–1998
- Unión de series y exportación de archivos finales (precios corrientes)
- Cálculo del deflactor implícito del PIB y deflactación a precios constantes base 2025

---
**Fuentes:**
- INE / BdE: `cntrb86.xls`, *Contabilidad Nacional Trimestral de España. Base 1986. Cuarto Trimestre de 1998.* Cobertura: 1970T1–1998T4. Unidad: miles de millones de pesetas.
  - Hoja 1: Demanda a precios constantes de 1986
  - Hoja 2: Demanda a precios corrientes
  - Hoja 3: Oferta a precios constantes de 1986
  - Hoja 4: Oferta a precios corrientes
- INE: `PIB pm Demanda (Precios corrientes) 1995-2026.csv`, Cobertura: 1995T1–2026T1. Unidad: millones de euros.
- INE: `PIB pm Oferta (Precios corrientes) 1995-2026.csv`, Cobertura: 1995T1–2026T1. Unidad: millones de euros.
- INE: `PIB pm Demanda (Indices de volumen encadenado) 1995-2026.csv`, Índices de volumen encadenado, datos no ajustados.
- INE: `PIB pm Oferta (Indices de volumen encadenado) 1995-2026.csv`, Índices de volumen encadenado, datos no ajustados.

# 0. Importación de librerías

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

# 1. Constantes globales

In [2]:
ruta = '../Fuentes/INE/CNT/'

# cntrb86 está en miles de millones de pesetas (mM pts)
# Tasa de conversión irrevocable: 1 EUR = 166,386 pts
# Conversión: mM pts × 1000 (→ M pts) ÷ 166.386 (→ M EUR)
factor_mmdpts_a_meur = 1000.0 / 166.386

# Fecha de corte: cntrb86 aporta hasta 1994T4; INE desde 1995T1
fecha_corte = pd.Timestamp('1995-01-01')

print(f'Factor conversión: {factor_mmdpts_a_meur:.6f} (miles M pts a M EUR)')

Factor conversión: 6.010121 (miles M pts a M EUR)


# 2. Carga de datasets
### 2.1 cntrb86.xls — Contabilidad Nacional Trimestral Base 1986 (1970–1998)
Unidad original: miles de millones de pesetas

In [3]:
xl_cntrb = pd.ExcelFile(ruta + 'cntrb86.xls', engine='xlrd')
print('Hojas disponibles:', xl_cntrb.sheet_names)

cntrb_hojas = {}
for hoja in xl_cntrb.sheet_names:
    df = xl_cntrb.parse(hoja, header=7)
    df = df[df['año'].notna() & (df['año'] != 'año')].reset_index(drop=True)
    cntrb_hojas[hoja] = df
    print(f'  Rango: {df["año"].min()} → {df["año"].max()}')
    print(f'  "{hoja}": {df.shape[0]} trimestres, {df.shape[1]} columnas')
    print(f'   Columnas: {df.columns.tolist()}')
    print()

Hojas disponibles: ['demanda a precios constantes 86', 'demanda a precios corrientes', 'oferta a precios constantes 86', 'oferta a precios corrientes']
  Rango: 1970 → 1998
  "demanda a precios constantes 86": 116 trimestres, 19 columnas
   Columnas: ['año', 'trim', 'Producto Interior Bruto p.m.', 'Consumo privado nacional', 'Consumo privado interior', 'Consumo público', 'Formación bruta de capital fijo', 'FBCF en bienes de equipo', 'FBCF en construcción', 'Variación de existencias', 'Demanda interna', 'Exportación de bienes y servicios', 'Exportación de bienes', 'Exportación de servicios', 'Consumo en el TE de no residentes', 'Importación de bienes y servicios', 'Importación de bienes', 'Importación de servicios', 'Consumo en el RM de residentes']

  Rango: 1970 → 1998
  "demanda a precios corrientes": 116 trimestres, 19 columnas
   Columnas: ['año', 'trim', 'Producto Interior Bruto p.m.', 'Consumo privado nacional', 'Consumo privado interior', 'Consumo público', 'Formación bruta de c

### 2.2 Archivos PIB del INE (1995–2026)
Precios corrientes e índices de volumen encadenado. Unidad corrientes: millones de euros. Datos no ajustados de estacionalidad.

In [4]:
archivos_pib = {
    'demanda_corrientes': 'PIB pm Demanda (Precios corrientes) 1995-2026.csv',
    'oferta_corrientes':  'PIB pm Oferta (Precios corrientes) 1995-2026.csv',
    'demanda_volumen':    'PIB pm Demanda (Indices de volumen encadenado) 1995-2026.csv',
    'oferta_volumen':     'PIB pm Oferta (Indices de volumen encadenado) 1995-2026.csv',
}

pib_95_26 = {}
for nombre, archivo in archivos_pib.items():
    try:
        df = pd.read_csv(ruta + archivo, sep=';', encoding='utf-8-sig')
    except UnicodeDecodeError:
        df = pd.read_csv(ruta + archivo, sep=';', encoding='latin1')
    pib_95_26[nombre] = df
    print(f'"{nombre}": {df.shape[0]} filas, {df.shape[1]} columnas')
    print(f'  Rango: {df["Periodo"].min()} → {df["Periodo"].max()}')
    print()

df_dem_raw = pib_95_26['demanda_corrientes']
df_of_raw  = pib_95_26['oferta_corrientes']
df_dem_vol_raw = pib_95_26['demanda_volumen']

"demanda_corrientes": 29250 filas, 10 columnas
  Rango: 1995T1 → 2026T1

"oferta_corrientes": 12000 filas, 6 columnas
  Rango: 1995T1 → 2026T1

"demanda_volumen": 46250 filas, 10 columnas
  Rango: 1995T1 → 2026T1

"oferta_volumen": 17500 filas, 6 columnas
  Rango: 1995T1 → 2026T1



# 3. Definición de categorías MECE

Solo se retienen las categorías disponibles de forma continua de **1970T1 a 2026T1** en ambas fuentes.

**Demanda:** `cntrb86` usa 'Demanda interna' que equivale a 'Demanda nacional' del INE (ambas = consumo privado + público + FBCF + variación existencias). Se renombran a `demanda_nacional` para coherencia.

**Oferta:** `cntrb86` reporta industria separada de construcción; el INE también (B-E vs F). La correspondencia es directa en el nivel sin construcción. Los `impuestos_netos` se calculan como residuo para evitar diferencias conceptuales entre bases (cntrb86 desglosa IVA e impuestos ligados a importaciones; el INE los unifica en 'Impuestos menos subvenciones sobre los productos').

**Categorías descartadas por disponibilidad parcial:**
- Demanda: FBCF bienes de equipo, FBCF construcción, consumo no residentes en TE, consumo residentes en RM (solo en cntrb86)
- Oferta: VAB ramas industriales totales, VAB ind. sin const. no energética, VAB servicios de mercado / no mercado (solo en cntrb86); sub-ramas de servicios por CNAE (solo en INE)

In [5]:
# ── Mappings cntrb86: nombre original de columna → nombre MECE unificado ──────
MECE_DEMANDA_CNTRB86 = {
    'Producto Interior Bruto p.m.'      : 'pib_pm',
    'Demanda interna'                   : 'demanda_nacional',
    'Consumo privado nacional'          : 'consumo_privado_nacional',
    'Consumo privado interior'          : 'consumo_privado_interior',
    'Consumo público'                   : 'consumo_publico',
    'Formación bruta de capital fijo'   : 'fbcf',
    'Variación de existencias'          : 'variacion_existencias',
    'Exportación de bienes y servicios' : 'exportacion_bienes_servicios',
    'Exportación de bienes'             : 'exportacion_bienes',
    'Exportación de servicios'          : 'exportacion_servicios',
    'Importación de bienes y servicios' : 'importacion_bienes_servicios',
    'Importación de bienes'             : 'importacion_bienes',
    'Importación de servicios'          : 'importacion_servicios',
}

MECE_OFERTA_CNTRB86 = {
    'Producto Interior Bruto p.m.'        : 'pib_pm',
    'VAB agricultura, ganadería, pesca'   : 'vab_agricultura',
    'VAB de la industria sin construcción': 'vab_industria',
    'VAB de la rama de la construcción'   : 'vab_construccion',
    'VAB de las ramas de los servicios'   : 'vab_servicios',
}

# ── Mappings INE: tupla de jerarquía (N1, N2, N3) → nombre MECE ──────────────
MECE_DEMANDA_INE = {
    ('Producto interior bruto a precios de mercado', '', '')                              : 'pib_pm',
    ('Demanda nacional', '', '')                                                          : 'demanda_nacional',
    ('Demanda nacional', 'Gasto en consumo final de los hogares y las ISFLSH', '')        : 'consumo_privado_nacional',
    ('Demanda nacional', 'Gasto en consumo final interior de los hogares', '')            : 'consumo_privado_interior',
    ('Demanda nacional', 'Gasto en consumo final de las AAPP', '')                        : 'consumo_publico',
    ('Demanda nacional', 'Formación bruta de capital', 'Formación bruta de capital fijo (FBCF)'): 'fbcf',
    ('Demanda nacional', 'Formación bruta de capital', 'Variación de existencias')        : 'variacion_existencias',
    ('Demanda externa', 'Exportaciones de bienes y servicios', '')                        : 'exportacion_bienes_servicios',
    ('Demanda externa', 'Exportaciones de bienes y servicios', 'Exportaciones de bienes'): 'exportacion_bienes',
    ('Demanda externa', 'Exportaciones de bienes y servicios', 'Exportaciones de servicios'): 'exportacion_servicios',
    ('Demanda externa', 'Importaciones de bienes y servicios', '')                        : 'importacion_bienes_servicios',
    ('Demanda externa', 'Importaciones de bienes y servicios', 'Importaciones de bienes'): 'importacion_bienes',
    ('Demanda externa', 'Importaciones de bienes y servicios', 'Importaciones de servicios'): 'importacion_servicios',
}

MECE_OFERTA_INE = {
    ('Producto interior bruto a precios de mercado', '')                                    : 'pib_pm',
    ('VABpb Agricultura, ganadería, silvicultura y pesca (A, CNAE 2009)', '')               : 'vab_agricultura',
    ('VABpb Industria (B-E, CNAE 2009)', '')                                                : 'vab_industria',
    ('VABpb Construcción (F, CNAE 2009)', '')                                               : 'vab_construccion',
    ('VABpb Servicios (G-T, CNAE 2009)', '')                                                : 'vab_servicios',
}

print(f'Columnas demanda cntrb86: {len(MECE_DEMANDA_CNTRB86)} columnas')
print(f'Columnas oferta  cntrb86: {len(MECE_OFERTA_CNTRB86)} columnas')
print(f'Columnas demanda INE:     {len(MECE_DEMANDA_INE)} columnas')
print(f'Columnas oferta  INE:     {len(MECE_OFERTA_INE)} columnas')

Columnas demanda cntrb86: 13 columnas
Columnas oferta  cntrb86: 5 columnas
Columnas demanda INE:     13 columnas
Columnas oferta  INE:     5 columnas


# 4. Limpieza y transformación
### 4.1 Función para cntrb86

In [6]:
def limpiar_cntrb_hoja(df_raw, mece_mapping):
    """
    Limpia una hoja de cntrb86 aplicando el mapping MECE:
    - Filtra y renombra columnas ANTES de cualquier normalización de nombres
    - Convierte año/trimestre a int y construye fecha
    - Convierte valores a float
    """
    df = df_raw.copy()

    # Convertir año y trimestre a entero
    df['año']  = df['año'].astype(int)
    df['trim'] = df['trim'].astype(int)

    # Construir fecha (primer día del trimestre)
    mes_inicio = (df['trim'] - 1) * 3 + 1
    df['fecha'] = pd.to_datetime(dict(year=df['año'], month=mes_inicio, day=1))
    df = df.rename(columns={'trim': 'trimestre'})

    # Verificar que las columnas MECE existen en la hoja
    missing = [c for c in mece_mapping if c not in df.columns]
    if missing:
        print(f'ALERTA  Columnas no encontradas en cntrb86: {missing}')

    # Filtrar solo las columnas MECE + tiempo (antes de snake_case)
    keep = [c for c in mece_mapping if c in df.columns]
    df = df[['fecha', 'año', 'trimestre'] + keep].copy()

    # Convertir valores a numérico
    for col in keep:
        df[col] = pd.to_numeric(
            df[col].astype(str).str.replace(',', '.', regex=False).str.strip(),
            errors='coerce'
        )

    # Renombrar a nombres MECE unificados
    df = df.rename(columns=mece_mapping)

    return df.sort_values('fecha').reset_index(drop=True)

In [7]:
# Corrientes (se convertirán a EUR más adelante)
cntrb_dem = limpiar_cntrb_hoja(cntrb_hojas['demanda a precios corrientes'], MECE_DEMANDA_CNTRB86)
cntrb_of  = limpiar_cntrb_hoja(cntrb_hojas['oferta a precios corrientes'],  MECE_OFERTA_CNTRB86)

# Constantes base 1986 (solo PIB pm, para el cálculo del deflactor)
cntrb_dem_const86 = limpiar_cntrb_hoja(cntrb_hojas['demanda a precios constantes 86'], MECE_DEMANDA_CNTRB86)

print('cntrb86 demanda corrientes:')
print(f'  Shape: {cntrb_dem.shape} | Rango: {cntrb_dem["fecha"].min().date()} → {cntrb_dem["fecha"].max().date()}')
print(f'  Columnas: {cntrb_dem.columns.tolist()}')
print()
print('cntrb86 oferta corrientes:')
print(f'  Shape: {cntrb_of.shape} | Rango: {cntrb_of["fecha"].min().date()} → {cntrb_of["fecha"].max().date()}')
print(f'  Columnas: {cntrb_of.columns.tolist()}')
print()
print('cntrb86 demanda constantes 86 (para deflactor):')
print(f'  Shape: {cntrb_dem_const86.shape} | Rango: {cntrb_dem_const86["fecha"].min().date()} → {cntrb_dem_const86["fecha"].max().date()}')

# Verificar nulos
for nombre, df in [('demanda cntrb86', cntrb_dem), ('oferta cntrb86', cntrb_of), ('constantes 86', cntrb_dem_const86)]:
    nulos = df.isna().sum()
    nulos_col = nulos[nulos > 0]
    if len(nulos_col) > 0:
        print(f'ALERTA  {nombre}: nulos en {nulos_col.to_dict()}')
    else:
        print(f'CORRECTO  {nombre}: sin valores nulos')

cntrb86 demanda corrientes:
  Shape: (116, 16) | Rango: 1970-01-01 → 1998-10-01
  Columnas: ['fecha', 'año', 'trimestre', 'pib_pm', 'demanda_nacional', 'consumo_privado_nacional', 'consumo_privado_interior', 'consumo_publico', 'fbcf', 'variacion_existencias', 'exportacion_bienes_servicios', 'exportacion_bienes', 'exportacion_servicios', 'importacion_bienes_servicios', 'importacion_bienes', 'importacion_servicios']

cntrb86 oferta corrientes:
  Shape: (116, 8) | Rango: 1970-01-01 → 1998-10-01
  Columnas: ['fecha', 'año', 'trimestre', 'pib_pm', 'vab_agricultura', 'vab_industria', 'vab_construccion', 'vab_servicios']

cntrb86 demanda constantes 86 (para deflactor):
  Shape: (116, 16) | Rango: 1970-01-01 → 1998-10-01
CORRECTO  demanda cntrb86: sin valores nulos
CORRECTO  oferta cntrb86: sin valores nulos
CORRECTO  constantes 86: sin valores nulos


### 4.2 Función para datos INE
Se usan los **datos no ajustados de estacionalidad y calendario**, filtrando por `Dato base`.

In [8]:
def limpiar_pib_ine(df_raw, nivel_cols, mece_mapping, ajustado=False):
    """
    Limpia y pivota un CSV del INE (formato largo → ancho).
    Filtra a datos base del tipo ajustado/no ajustado y conserva
    solo los agregados del mapping MECE.
    """
    df = df_raw.copy()

    tipo_dato = (
        'Datos ajustados de estacionalidad y calendario'
        if ajustado else
        'Datos no ajustados de estacionalidad y calendario'
    )
    filtro = (
        (df['Tipo de dato'] == tipo_dato) &
        (df['Niveles y tasas'] == 'Dato base')
    )
    df = df[filtro].copy()

    # Rellenar NaN en columnas de jerarquía
    for col in nivel_cols:
        df[col] = df[col].fillna('').astype(str).str.strip()

    # Clave de jerarquía como tupla
    df['clave'] = df[nivel_cols].apply(tuple, axis=1)

    # Filtrar solo agregados MECE
    df = df[df['clave'].isin(mece_mapping.keys())].copy()
    df['agregado'] = df['clave'].map(mece_mapping)

    # Parsear período trimestral
    s = df['Periodo'].astype(str).str.strip()
    df['año']       = s.str[:4].astype(int)
    df['trimestre'] = s.str[5:].astype(int)
    mes_inicio      = (df['trimestre'] - 1) * 3 + 1
    df['fecha']     = pd.to_datetime(dict(year=df['año'], month=mes_inicio, day=1))

    # Limpiar Total (formato numérico español)
    df['Total'] = (
        df['Total'].astype(str).str.strip()
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
    )
    df['Total'] = pd.to_numeric(df['Total'], errors='coerce')

    # Pivot a formato ancho
    df_pivot = df.pivot_table(
        index=['fecha', 'año', 'trimestre'],
        columns='agregado',
        values='Total',
        aggfunc='first'
    ).reset_index()
    df_pivot.columns.name = None
    return df_pivot.sort_values('fecha').reset_index(drop=True)

In [9]:
niveles_demanda = [
    'Agregados macroeconómicos: Nivel 1',
    'Agregados macroeconómicos: Nivel 2',
    'Agregados macroeconómicos: Nivel 3',
]
niveles_oferta = [
    'Agregados macroeconómicos: Nivel 1',
    'Agregados macroeconómicos: Nivel 2',
]

# Datos no ajustados de estacionalidad (1995-2026)
ine_dem = limpiar_pib_ine(df_dem_raw, niveles_demanda, MECE_DEMANDA_INE, ajustado=False)
ine_of  = limpiar_pib_ine(df_of_raw,  niveles_oferta,  MECE_OFERTA_INE,  ajustado=False)

# Índices de volumen encadenado (solo PIB pm, para el deflactor)
MECE_PIB_PM_VOL = {
    ('Producto interior bruto a precios de mercado', '', ''): 'pib_pm',
}
ine_vol = limpiar_pib_ine(df_dem_vol_raw, niveles_demanda, MECE_PIB_PM_VOL, ajustado=False)

print(f'INE demanda corrientes: {ine_dem.shape} | {ine_dem["fecha"].min().date()} → {ine_dem["fecha"].max().date()}')
print(f'  Columnas: {[c for c in ine_dem.columns if c not in ["fecha","año","trimestre"]]}')
print()
print(f'INE oferta corrientes:  {ine_of.shape}  | {ine_of["fecha"].min().date()} → {ine_of["fecha"].max().date()}')
print(f'  Columnas: {[c for c in ine_of.columns if c not in ["fecha","año","trimestre"]]}')
print()
print(f'INE volumen encadenado (PIB pm): {ine_vol.shape} | {ine_vol["fecha"].min().date()} → {ine_vol["fecha"].max().date()}')

# Verificar nulos
for nombre, df in [('INE demanda', ine_dem), ('INE oferta', ine_of), ('INE volumen', ine_vol)]:
    nulos = df.isna().sum()
    nulos_col = nulos[nulos > 0]
    if len(nulos_col) > 0:
        print(f'ALERTA  {nombre}: nulos en {nulos_col.to_dict()}')
    else:
        print(f'CORRECTO  {nombre}: sin valores nulos')

INE demanda corrientes: (125, 16) | 1995-01-01 → 2026-01-01
  Columnas: ['consumo_privado_interior', 'consumo_privado_nacional', 'consumo_publico', 'demanda_nacional', 'exportacion_bienes', 'exportacion_bienes_servicios', 'exportacion_servicios', 'fbcf', 'importacion_bienes', 'importacion_bienes_servicios', 'importacion_servicios', 'pib_pm', 'variacion_existencias']

INE oferta corrientes:  (125, 8)  | 1995-01-01 → 2026-01-01
  Columnas: ['pib_pm', 'vab_agricultura', 'vab_construccion', 'vab_industria', 'vab_servicios']

INE volumen encadenado (PIB pm): (125, 4) | 1995-01-01 → 2026-01-01
CORRECTO  INE demanda: sin valores nulos
CORRECTO  INE oferta: sin valores nulos
CORRECTO  INE volumen: sin valores nulos


### 4.3 Conversión de unidades: pesetas → euros

`cntrb86` está en **miles de millones de pesetas**.
Los CSVs del INE están en **millones de euros**.

Factor de conversión: **1 EUR = 166,386 pesetas** (tipo de conversión irrevocable fijado el 31/12/1998).
$$\text{M EUR} = \text{mM pts} \times \frac{1000}{166{,}386} \approx 6{,}0101$$

In [10]:
cols_valor_dem = [c for c in cntrb_dem.columns if c not in ['fecha', 'año', 'trimestre']]
cols_valor_of  = [c for c in cntrb_of.columns  if c not in ['fecha', 'año', 'trimestre']]

cntrb_dem_eur = cntrb_dem.copy()
cntrb_dem_eur[cols_valor_dem] = cntrb_dem_eur[cols_valor_dem] * factor_mmdpts_a_meur

cntrb_of_eur = cntrb_of.copy()
cntrb_of_eur[cols_valor_of] = cntrb_of_eur[cols_valor_of] * factor_mmdpts_a_meur

# Verificación puntual: PIB 1970T1
pib_1970 = cntrb_dem[cntrb_dem['año'] == 1970].iloc[0]
pib_1970_eur = cntrb_dem_eur[cntrb_dem_eur['año'] == 1970].iloc[0]
print(f'PIB 1970T1 original: {pib_1970["pib_pm"]:.2f} miles M pts')
print(f'PIB 1970T1 convertido: {pib_1970_eur["pib_pm"]:.2f} M EUR')
print(f'(× {factor_mmdpts_a_meur:.6f})')
print()
# Verificación: PIB en período de solapamiento (1995T1), para comparar con INE
pib_1995_b86 = cntrb_dem_eur[cntrb_dem_eur['fecha'] == '1995-01-01'].iloc[0]['pib_pm']
pib_1995_ine = ine_dem[ine_dem['fecha'] == '1995-01-01'].iloc[0]['pib_pm']
print(f'PIB 1995T1 — cntrb86 (en M EUR): {pib_1995_b86:.0f}')
print(f'PIB 1995T1 — INE (M EUR): {pib_1995_ine:.0f}')
print(f'Diferencia: {(pib_1995_b86 - pib_1995_ine) / pib_1995_ine * 100:.1f}% (revisiones metodológicas)')

PIB 1970T1 original: 642.70 miles M pts
PIB 1970T1 convertido: 3862.70 M EUR
(× 6.010121)

PIB 1995T1 — cntrb86 (en M EUR): 102363
PIB 1995T1 — INE (M EUR): 109022
Diferencia: -6.1% (revisiones metodológicas)


# 5. Impuestos netos sobre los productos (oferta, cálculo residual)

Cntrb86 desglosa dos conceptos distintos (impuestos netos ligados a importaciones + IVA sobre los productos) mientras que el INE los unifica en impuestos menos subvenciones sobre los productos. Para garantizar coherenciase calcula como residuo del pib menos los valores agregados brutos:

$$\text{impuestos\_netos} = \text{pib\_pm} - \text{vab\_agricultura} - \text{vab\_industria} - \text{vab\_construccion} - \text{vab\_servicios}$$

In [11]:
cols_vab = ['vab_agricultura', 'vab_industria', 'vab_construccion', 'vab_servicios']

for df, nombre in [(cntrb_of_eur, 'df_70_98'), (ine_of, 'df_95_26')]:
    df['impuestos_netos'] = df['pib_pm'] - df[cols_vab].sum(axis=1)
    residuo_medio = df['impuestos_netos'].mean()
    pct_medio     = (df['impuestos_netos'] / df['pib_pm'] * 100).mean()
    print(f'  {nombre}: impuestos_netos medio = {residuo_medio:.0f} M EUR ({pct_medio:.1f}% del PIB)')

print()

# Verificación  con umbrales históricos .
# España introduce el IVA el 01/01/1986 al adherirse a la CEE, sustituyendo al IGTE (Impuesto General sobre el tráfico de Empresas)
UMBRALES = {
    'pre_iva':  {'fecha_min': '1900-01-01', 'fecha_max': '1985-12-31',
                 'min_pct': 2.0, 'max_pct': 8.0, 'nota': 'era IGTE (pre-IVA)'},
    'post_iva': {'fecha_min': '1986-01-01', 'fecha_max': '2099-12-31',
                 'min_pct': 5.0, 'max_pct': 15.0, 'nota': 'era IVA (post-1985)'},
}

for df, nombre in [(cntrb_of_eur, 'df_70_98'), (ine_of, 'df_95_26')]:
    pct_serie = df['impuestos_netos'] / df['pib_pm'] * 100
    errores = []

    for periodo, u in UMBRALES.items():
        mask = (df['fecha'] >= pd.Timestamp(u['fecha_min'])) & \
               (df['fecha'] <= pd.Timestamp(u['fecha_max']))
        if not mask.any():
            continue
        fuera = df[mask & ((pct_serie < u['min_pct']) | (pct_serie > u['max_pct']))]
        if not fuera.empty:
            errores.append((periodo, u['nota'], fuera, pct_serie[fuera.index]))

    if not errores:
        resumen = pct_serie.agg(['min', 'mean', 'max']).round(1)
        print(f'CORRECTO  {nombre}: impuestos_netos dentro de rangos históricos '
              f'(min={resumen["min"]}%, media={resumen["mean"]}%, max={resumen["max"]}%)')
    else:
        for periodo, nota, fuera, pcts in errores:
            print(f'ALERTA  {nombre} [{nota}]: {len(fuera)} trimestres fuera de rango:')
            print(fuera[['fecha', 'impuestos_netos']].assign(
                pct_pib=pcts.round(1)).to_string(index=False))

  df_70_98: impuestos_netos medio = 2804 M EUR (5.0% del PIB)
  df_95_26: impuestos_netos medio = 22524 M EUR (8.9% del PIB)

CORRECTO  df_70_98: impuestos_netos dentro de rangos históricos (min=3.5%, media=5.0%, max=6.8%)
CORRECTO  df_95_26: impuestos_netos dentro de rangos históricos (min=5.3%, media=8.9%, max=12.3%)


# 6. Verificación del solapamiento (1995T1–1998T4)

Ambas fuentes se solapan 16 trimestres. Las diferencias son esperables: el INE incorpora revisiones metodológicas posteriores a 1998. El dataset final usa la fuente más reciente a partir de 1995T1.

In [12]:
sol_ini = pd.Timestamp('1995-01-01')
sol_fin = pd.Timestamp('1998-10-01')

df_solap_b86 = cntrb_dem_eur[
    (cntrb_dem_eur['fecha'] >= sol_ini) & (cntrb_dem_eur['fecha'] <= sol_fin)
][['fecha', 'pib_pm']].rename(columns={'pib_pm': 'pib_b86_meur'})

df_solap_ine = ine_dem[
    (ine_dem['fecha'] >= sol_ini) & (ine_dem['fecha'] <= sol_fin)
][['fecha', 'pib_pm']].rename(columns={'pib_pm': 'pib_ine_meur'})

df_comp = pd.merge(df_solap_b86, df_solap_ine, on='fecha')
df_comp['dif_pct'] = (
    (df_comp['pib_b86_meur'] - df_comp['pib_ine_meur'])
    / df_comp['pib_ine_meur'] * 100
).round(2)

print('PIB p.m. en solapamiento 1995–1998 (M EUR, precios corrientes):')
print(df_comp.to_string(index=False))
print()
print(f'Diferencia media absoluta: {df_comp["dif_pct"].abs().mean():.1f}%')

PIB p.m. en solapamiento 1995–1998 (M EUR, precios corrientes):
     fecha  pib_b86_meur  pib_ine_meur  dif_pct
1995-01-01 102363.095453        109022    -6.11
1995-04-01 104266.074069        116592   -10.57
1995-07-01 105738.187107        112289    -5.83
1995-10-01 107019.238397        122356   -12.53
1996-01-01 108636.898537        115660    -6.07
1996-04-01 110048.519707        123277   -10.73
1996-07-01 111549.289003        120083    -7.11
1996-10-01 112971.217530        129926   -13.05
1997-01-01 114488.124001        121821    -6.02
1997-04-01 115979.102809        130462   -11.10
1997-07-01 117950.344380        127063    -7.17
1997-10-01 119750.339572        139625   -14.23
1998-01-01 121576.611013        130326    -6.71
1998-04-01 123241.564795        139959   -11.94
1998-07-01 125054.157201        136326    -8.27
1998-10-01 126865.751926        148948   -14.83

Diferencia media absoluta: 9.5%


In [13]:
fig = px.line(
    df_comp.melt('fecha', value_vars=['pib_b86_meur', 'pib_ine_meur']),
    x='fecha', y='value', color='variable',
    title='PIB p.m. en solapamiento 1995–1998: cntrb86 vs INE (M EUR, precios corrientes)',
    labels={'value': 'Millones EUR', 'fecha': 'Trimestre', 'variable': 'Fuente'}
)
fig.show()

# 7. Unión de series con enlace retrospectivo (precios corrientes)

**Estrategia:** enlace retrospectivo por coeficiente único anclado en el PIB.
- cntrb86 (M EUR, precios corrientes): aporta 1970T1–1994T4 (100 trimestres)
- INE (M EUR, precios corrientes): aporta 1995T1–2026T1 (125 trimestres)

Las dos fuentes difieren en nivel en el solapamiento (cntrb86 aprox 6,5 % por debajo del INE en 1995T1) por revisiones metodológicas. Una concatenación cruda dejaría un salto en la costura 1994T4→1995T1 que produce un pico espurio en la variación interanual del PIB de 1995–96: no es crecimiento real, es el cambio de base disfrazado de economía.

Para corregirlo se reescala todo el bloque histórico (pre-1995) por el ratio INE/cntrb86 del PIB p.m. en 1995T1. Al ser un factor común y lineal, este enlace preserva exactamente las identidades contables (los componentes siguen sumando el PIB), no altera las tasas de crecimiento de ninguna serie, mantiene fieles los shares/ratios históricos (el factor se cancela en el cociente) y hace el PIB continuo en la costura, eliminando el pico de var% de raíz. Equivale a la retropolación por tasas anclada en 1995T1.


**Limitación documentada (estacionalidad heterogénea):** el enlace corrige el nivel pero no la forma estacional. cntrb86 (pre-1995) tiene una estacionalidad trimestral mucho más suave (amplitud intra-anual aprox 1,9 %, perfil casi plano) que el INE (aprox 8,1 %, pico en T4). Esta inhomogeneidad, inherente al cambio de fuente y no corregible sin reestimar la estacionalidad de los años 70-80, deja un residuo menor en la variación interanual de 1995 (único año con numerador INE y denominador cntrb86) y un cambio de régimen estacional del nivel hacia 1995, relevante para los modelos en niveles (Prophet/VECM). La forma estacionaria adoptada para el PIB es la variación interanual, que neutraliza la estacionalidad salvo en ese año de transición.

Total: **225 trimestres**.

In [14]:
# ── 7.0  Enlace retrospectivo por coeficiente único (anclado en el PIB) ──────
# Corrige el salto de nivel del empalme (cntrb86 aprox 6,5% por debajo del INE en 1995T1)
# que generaba el pico espurio de la variación interanual del PIB en 1995-96.
# Se reescala el bloque histórico completo cntrb86 por el ratio INE/cntrb86 del PIB en 1995T1.
# Factor común y lineal => preserva identidades contables, no altera tasas de crecimiento,
# mantiene fieles los shares/ratios y deja el PIB continuo en la costura.
# Equivale a la retropolación por tasas anclada en 1995T1 (idénticas para un único punto).

pib_ine_1995 = ine_dem.loc[ine_dem['fecha'] == fecha_corte, 'pib_pm'].iloc[0]
pib_b86_1995 = cntrb_dem_eur.loc[cntrb_dem_eur['fecha'] == fecha_corte, 'pib_pm'].iloc[0]
coef_enlace_pib = pib_ine_1995 / pib_b86_1995

print(f'PIB p.m. 1995T1 — INE: {pib_ine_1995:.0f} | cntrb86: {pib_b86_1995:.0f}')
print(f'Coeficiente de enlace (único, anclado en el PIB): {coef_enlace_pib:.5f}')

# Aplicar a todas las columnas de valor del bloque histórico (demanda y oferta).
# Solo el tramo pre-1995 alimenta la unión; el solape 1995-1998 de cntrb86 se descarta en unir_series().
for _df in (cntrb_dem_eur, cntrb_of_eur):
    _cols = [c for c in _df.columns if c not in ('fecha', 'año', 'trimestre')]
    _df[_cols] = _df[_cols] * coef_enlace_pib

# Verificación 1: el PIB cntrb86 reescalado coincide con INE en 1995T1 (continuidad en la costura)
pib_b86_post = cntrb_dem_eur.loc[cntrb_dem_eur['fecha'] == fecha_corte, 'pib_pm'].iloc[0]
print(f'PIB cntrb86 reescalado en 1995T1: {pib_b86_post:.0f} (≈ INE {pib_ine_1995:.0f})')

# Verificación 2: la identidad de oferta (PIB = ΣVAB + impuestos_netos) se mantiene tras el reescalado
_cols_of = ['vab_agricultura', 'vab_industria', 'vab_construccion', 'vab_servicios', 'impuestos_netos']
_resid = (cntrb_of_eur['pib_pm'] - cntrb_of_eur[_cols_of].sum(axis=1)).abs().max()
print(f'Identidad oferta tras enlace (max |PIB - ΣVAB - imp_netos|): {_resid:.6f}')


PIB p.m. 1995T1 — INE: 109022 | cntrb86: 102363
Coeficiente de enlace (único, anclado en el PIB): 1.06505
PIB cntrb86 reescalado en 1995T1: 109022 (≈ INE 109022)
Identidad oferta tras enlace (max |PIB - ΣVAB - imp_netos|): 0.000000


In [15]:
def unir_series(df_hist, df_nuevo, fecha_corte):
    parte_hist  = df_hist[df_hist['fecha'] < fecha_corte].copy()
    parte_nuevo = df_nuevo[df_nuevo['fecha'] >= fecha_corte].copy()


    df_union = pd.concat([parte_hist, parte_nuevo], ignore_index=True)
    return df_union.sort_values('fecha').reset_index(drop=True)


df_dem_corrientes = unir_series(
    cntrb_dem_eur, ine_dem, fecha_corte
)

df_of_corrientes = unir_series(
    cntrb_of_eur, ine_of, fecha_corte,
)

print(f'Demanda corrientes: {df_dem_corrientes.shape}')
print(f'  Rango: {df_dem_corrientes["fecha"].min().date()} → {df_dem_corrientes["fecha"].max().date()}')
print()
print(f'Oferta corrientes: {df_of_corrientes.shape}')
print(f'  Rango: {df_of_corrientes["fecha"].min().date()} → {df_of_corrientes["fecha"].max().date()}')

# Verificar ausencia de gaps temporales
for nombre, df in [('demanda', df_dem_corrientes), ('oferta', df_of_corrientes)]:
    esperados = pd.date_range(df['fecha'].min(), df['fecha'].max(), freq='QS')
    encontrados = pd.DatetimeIndex(df['fecha'])
    gaps = esperados.difference(encontrados)
    if len(gaps) > 0:
        print(f'ALERTA  {nombre}: gaps en {gaps.tolist()}')
    else:
        print(f'CORRECTO  {nombre}: serie temporal continua, sin huecos')

Demanda corrientes: (225, 16)
  Rango: 1970-01-01 → 2026-01-01

Oferta corrientes: (225, 9)
  Rango: 1970-01-01 → 2026-01-01
CORRECTO  demanda: serie temporal continua, sin huecos
CORRECTO  oferta: serie temporal continua, sin huecos


# 8. Deflactor implícito del PIB

El deflactor implícito del PIB se calcula como el ratio entre el PIB a precios corrientes (nominal) y el PIB en volumen (real):

$$D_t = \frac{\text{PIB nominal}_t}{\text{PIB volumen}_t}$$

- **1995–2026:** Se usa el ratio entre `PIB corrientes (M EUR)` y el `índice de volumen encadenado` del INE.
- **1970–1994:** Se usa el ratio entre `PIB corrientes` y `PIB constantes base 1986` de cntrb86.
- **Empalme:** Se enlazan ambas series en 1995T1 mediante un factor de escala que iguala los niveles.
- **Rebasado:** Se normaliza para que la media de los trimestres de 2025 sea 100.

El deflactor se aplica a **todas** las componentes de demanda y oferta para obtener precios constantes base 2025:

$$\text{Valor constante 2025}_t = \frac{\text{Valor corriente}_t}{D^{2025}_t / 100}$$

In [16]:
# ── 8.1  Deflactor INE (1995–2026) ──────────────────────────────────────────
# Ratio nominal / volumen encadenado
deflactor_ine = ine_dem[['fecha', 'año', 'trimestre']].copy()
deflactor_ine['d_raw'] = ine_dem['pib_pm'].values / ine_vol['pib_pm'].values

# ── 8.2  Deflactor cntrb86 (1970–1998) ─────────────────────────────────────
# Ratio corrientes / constantes base 86 (ambos en miles mM pts, se cancelan)
deflactor_86 = cntrb_dem[['fecha', 'año', 'trimestre']].copy()
deflactor_86['d_raw'] = cntrb_dem['pib_pm'].values / cntrb_dem_const86['pib_pm'].values

# ── 8.3  Empalme en 1995T1 ─────────────────────────────────────────────────
# Factor de enlace: iguala el nivel del deflactor 86 al del INE en el punto de corte
d_ine_1995 = deflactor_ine.loc[deflactor_ine['fecha'] == fecha_corte, 'd_raw'].iloc[0]
d_86_1995  = deflactor_86.loc[deflactor_86['fecha'] == fecha_corte, 'd_raw'].iloc[0]
factor_enlace = d_ine_1995 / d_86_1995

print(f'Deflactor INE en 1995T1:    {d_ine_1995:.4f}')
print(f'Deflactor cntrb86 en 1995T1: {d_86_1995:.4f}')
print(f'Factor de enlace:            {factor_enlace:.4f}')

# Escalar la serie histórica
deflactor_86['d_raw'] = deflactor_86['d_raw'] * factor_enlace

# ── 8.4  Serie unificada ───────────────────────────────────────────────────
parte_hist  = deflactor_86[deflactor_86['fecha'] < fecha_corte][['fecha', 'd_raw']].copy()
parte_nueva = deflactor_ine[deflactor_ine['fecha'] >= fecha_corte][['fecha', 'd_raw']].copy()
deflactor = pd.concat([parte_hist, parte_nueva], ignore_index=True).sort_values('fecha').reset_index(drop=True)

# ── 8.5  Rebasar a 2025 = 100 ─────────────────────────────────────────────
d_2025 = deflactor.loc[
    (deflactor['fecha'] >= '2025-01-01') & (deflactor['fecha'] <= '2025-10-01'), 'd_raw'
].mean()
deflactor['deflactor_2025'] = deflactor['d_raw'] / d_2025 * 100

print(f'\nMedia del deflactor bruto en 2025: {d_2025:.4f}')
print(f'Deflactor base 2025 — rango: {deflactor["deflactor_2025"].min():.2f} → {deflactor["deflactor_2025"].max():.2f}')
print(f'Trimestres: {len(deflactor)}')

# Verificación: el deflactor en 2025 debe promediar ~100
check_2025 = deflactor.loc[
    (deflactor['fecha'] >= '2025-01-01') & (deflactor['fecha'] <= '2025-10-01'), 'deflactor_2025'
]
print(f'Media del deflactor en 2025: {check_2025.mean():.2f} (debe ser 100.00)')

Deflactor INE en 1995T1:    1691.8032
Deflactor cntrb86 en 1995T1: 1.6470
Factor de enlace:            1027.1870

Media del deflactor bruto en 2025: 3408.4827
Deflactor base 2025 — rango: 3.80 → 102.99
Trimestres: 225
Media del deflactor en 2025: 100.00 (debe ser 100.00)


In [17]:
fig = px.line(
    deflactor, x='fecha', y='deflactor_2025',
    title='Deflactor implícito del PIB (base 2025 = 100) — 1970T1 a 2026T1',
    labels={'deflactor_2025': 'Índice (2025=100)', 'fecha': 'Trimestre'}
)
fig.add_shape(type='line', x0='1995-01-01', x1='1995-01-01', y0=0, y1=1,
              yref='paper', line=dict(dash='dash', color='red'))
fig.add_annotation(x='1995-01-01', y=1, yref='paper', text='Empalme 1995T1',
                   showarrow=False, yshift=10)
fig.show()

# 9. Deflactación: precios constantes base 2025

Se divide cada valor nominal (precios corrientes) entre el deflactor del PIB rebasado a 2025, para obtener valores en **millones de euros a precios constantes de 2025**.

$$\text{Valor real}_t = \frac{\text{Valor nominal}_t}{D^{2025}_t / 100}$$

In [18]:
# Merge deflactor con las series unificadas de corrientes
deflactor_merge = deflactor[['fecha', 'deflactor_2025']].copy()

# ── Demanda constantes base 2025 ──
df_dem_constantes = df_dem_corrientes.merge(deflactor_merge, on='fecha', how='left')
cols_valor_dem_defl = [c for c in df_dem_constantes.columns
                      if c not in ['fecha', 'año', 'trimestre', 'fuente', 'deflactor_2025']]
for col in cols_valor_dem_defl:
    df_dem_constantes[col] = df_dem_constantes[col] / (df_dem_constantes['deflactor_2025'] / 100)
df_dem_constantes = df_dem_constantes.drop(columns='deflactor_2025')

# ── Oferta constantes base 2025 ──
df_of_constantes = df_of_corrientes.merge(deflactor_merge, on='fecha', how='left')
cols_valor_of_defl = [c for c in df_of_constantes.columns
                     if c not in ['fecha', 'año', 'trimestre', 'fuente', 'deflactor_2025']]
for col in cols_valor_of_defl:
    df_of_constantes[col] = df_of_constantes[col] / (df_of_constantes['deflactor_2025'] / 100)
df_of_constantes = df_of_constantes.drop(columns='deflactor_2025')

print(f'Demanda constantes 2025: {df_dem_constantes.shape}')
print(f'Oferta constantes 2025:  {df_of_constantes.shape}')
print()

# Verificación: en 2025, los valores constantes deben ser ≈ a los corrientes
for nombre, df_c, df_n in [('demanda', df_dem_constantes, df_dem_corrientes),
                             ('oferta', df_of_constantes, df_of_corrientes)]:
    mask_2025 = df_c['año'] == 2025
    pib_const = df_c.loc[mask_2025, 'pib_pm'].mean()
    pib_nom   = df_n.loc[df_n['año'] == 2025, 'pib_pm'].mean()
    print(f'  {nombre} 2025 — PIB corrientes: {pib_nom:.0f} | PIB constantes 2025: {pib_const:.0f} | ratio: {pib_const/pib_nom:.4f}')

Demanda constantes 2025: (225, 16)
Oferta constantes 2025:  (225, 9)

  demanda 2025 — PIB corrientes: 421788 | PIB constantes 2025: 421586 | ratio: 0.9995
  oferta 2025 — PIB corrientes: 421788 | PIB constantes 2025: 421586 | ratio: 0.9995


# 10. Visualización de los datasets finales

In [19]:
# PIB p.m. serie completa 1970-2026: corrientes vs constantes 2025
df_viz = pd.DataFrame({
    'fecha': df_dem_corrientes['fecha'],
    'PIB corrientes': df_dem_corrientes['pib_pm'],
    'PIB constantes 2025': df_dem_constantes['pib_pm'],
})
fig = px.line(
    df_viz.melt('fecha'),
    x='fecha', y='value', color='variable',
    title='PIB p.m. (M EUR) — 1970T1 a 2026T1',
    labels={'value': 'Millones EUR', 'fecha': 'Trimestre', 'variable': ''}
)
fig.show()

In [20]:
# Demanda: componentes principales (corrientes)
cols_dem_plot = ['consumo_privado_nacional', 'consumo_publico', 'fbcf', 'exportacion_bienes_servicios', 'importacion_bienes_servicios']
fig2 = px.line(
    df_dem_corrientes.melt('fecha', value_vars=cols_dem_plot),
    x='fecha', y='value', color='variable',
    title='Componentes de demanda — precios corrientes (M EUR) — 1970–2026',
    labels={'value': 'Millones EUR', 'fecha': 'Trimestre', 'variable': 'Componente'}
)
fig2.show()

In [21]:
# Oferta: composición del PIB por sectores (corrientes)
cols_of_plot = ['vab_agricultura', 'vab_industria', 'vab_construccion', 'vab_servicios', 'impuestos_netos']
fig3 = px.area(
    df_of_corrientes, x='fecha', y=cols_of_plot,
    title='Composición del PIB por oferta — precios corrientes (M EUR) — 1970–2026',
    labels={'value': 'Millones EUR', 'fecha': 'Trimestre', 'variable': 'Sector'}
)
fig3.show()

# 11. Exportación de archivos finales

Cuatro archivos CSV, separador `;`, codificación `utf-8-sig`:
1. `pib_demanda_corrientes_1970_2026.csv` — Precios corrientes, millones de euros
2. `pib_oferta_corrientes_1970_2026.csv` — Precios corrientes, millones de euros
3. `pib_demanda_constantes_2025_1970_2026.csv` — Precios constantes base 2025, millones de euros
4. `pib_oferta_constantes_2025_1970_2026.csv` — Precios constantes base 2025, millones de euros

In [22]:
# ── Recortar al rango de tasa de paro y guardar en Datasets ────────────
from pathlib import Path
FECHA_INICIO = '1974-07-01'
FECHA_FIN    = '2025-10-01'
ruta_datasets = Path(r'C:\Users\marco\PycharmProjects\TFM_Marcos\Datasets')

time_cols = ['fecha', 'año', 'trimestre']

dem_val_cols = [
    'pib_pm',
    'demanda_nacional',
    'consumo_privado_nacional', 'consumo_privado_interior', 'consumo_publico',
    'fbcf', 'variacion_existencias',
    'exportacion_bienes_servicios', 'exportacion_bienes', 'exportacion_servicios',
    'importacion_bienes_servicios', 'importacion_bienes', 'importacion_servicios',
]

of_val_cols = [
    'pib_pm',
    'vab_agricultura', 'vab_industria', 'vab_construccion', 'vab_servicios',
    'impuestos_netos',
]

# Corrientes
df_dem_corrientes = df_dem_corrientes[time_cols + [c for c in dem_val_cols if c in df_dem_corrientes.columns]]
df_of_corrientes  = df_of_corrientes[time_cols  + [c for c in of_val_cols  if c in df_of_corrientes.columns]]


df_dem_corrientes = df_dem_corrientes[(df_dem_corrientes['fecha'] >= FECHA_INICIO) & (df_dem_corrientes['fecha'] <= FECHA_FIN)].copy()
df_dem_corrientes.to_csv(ruta_datasets / 'pib_demanda_corrientes_1970_2026.csv', index=False, encoding='utf-8-sig')
df_of_corrientes = df_of_corrientes[(df_of_corrientes['fecha'] >= FECHA_INICIO) & (df_of_corrientes['fecha'] <= FECHA_FIN)].copy()
df_of_corrientes.to_csv( ruta_datasets / 'pib_oferta_corrientes_1970_2026.csv', index=False, encoding='utf-8-sig')

# Constantes,  base 2025
df_dem_constantes = df_dem_constantes[time_cols + [c for c in dem_val_cols if c in df_dem_constantes.columns]]
df_of_constantes  = df_of_constantes[time_cols  + [c for c in of_val_cols  if c in df_of_constantes.columns]]

df_dem_constantes = df_dem_constantes[(df_dem_constantes['fecha'] >= FECHA_INICIO) & (df_dem_constantes['fecha'] <= FECHA_FIN)].copy()
df_dem_constantes.to_csv(ruta_datasets / 'pib_demanda_constantes_2025_1970_2026.csv', index=False, encoding='utf-8-sig')
df_of_constantes = df_of_constantes[(df_of_constantes['fecha'] >= FECHA_INICIO) & (df_of_constantes['fecha'] <= FECHA_FIN)].copy()
df_of_constantes.to_csv( ruta_datasets / 'pib_oferta_constantes_2025_1970_2026.csv', index=False, encoding='utf-8-sig')



# 12. Información acerca de los datasets finales

In [23]:
# Vista previa de los archivos exportados
for nombre, df in [('DEMANDA CORRIENTES', df_dem_corrientes),
                    ('OFERTA CORRIENTES', df_of_corrientes),
                    ('DEMANDA CONSTANTES 2025', df_dem_constantes),
                    ('OFERTA CONSTANTES 2025', df_of_constantes)]:
    print(f'Información acerca de {nombre}:')
    print(f'{df.shape[0]} trimestres × {df.shape[1]} columnas | Rango: {df["fecha"].min().date()} → {df["fecha"].max().date()}')
    columnas = df.columns.tolist()
    print(f'  Columnas: {columnas}')
    print()
    nulos = df.isna().sum()
    nulos_col = nulos[nulos > 0]
    if len(nulos_col) > 0:
        print(f'ALERTA  {nombre}: nulos en {nulos_col.to_dict()}')
    else:
        print(f'CORRECTO  {nombre}: sin valores nulos')
    print()
    print(f'=== {nombre} (primeras y últimas 2 filas) ===')
    print(pd.concat([df.head(2), df.tail(2)]).to_string(index=False))
    print()


Información acerca de DEMANDA CORRIENTES:
206 trimestres × 16 columnas | Rango: 1974-07-01 → 2025-10-01
  Columnas: ['fecha', 'año', 'trimestre', 'pib_pm', 'demanda_nacional', 'consumo_privado_nacional', 'consumo_privado_interior', 'consumo_publico', 'fbcf', 'variacion_existencias', 'exportacion_bienes_servicios', 'exportacion_bienes', 'exportacion_servicios', 'importacion_bienes_servicios', 'importacion_bienes', 'importacion_servicios']

CORRECTO  DEMANDA CORRIENTES: sin valores nulos

=== DEMANDA CORRIENTES (primeras y últimas 2 filas) ===
     fecha  año  trimestre        pib_pm  demanda_nacional  consumo_privado_nacional  consumo_privado_interior  consumo_publico         fbcf  variacion_existencias  exportacion_bienes_servicios  exportacion_bienes  exportacion_servicios  importacion_bienes_servicios  importacion_bienes  importacion_servicios
1974-07-01 1974          3   8358.543702       8832.397213               5449.248165               5714.253303       830.221411  2367.123190  

In [24]:
for nombre, df in [('DEMANDA CORRIENTES', df_dem_corrientes),
                    ('OFERTA CORRIENTES', df_of_corrientes),
                    ('DEMANDA CONSTANTES 2025', df_dem_constantes),
                    ('OFERTA CONSTANTES 2025', df_of_constantes)]:
    print(f'Información acerca de {nombre}:')
    print(f'{df.describe()}')

Información acerca de DEMANDA CORRIENTES:
                               fecha          año   trimestre         pib_pm  \
count                            206   206.000000  206.000000     206.000000   
mean   2000-02-15 05:56:30.291262080  1999.747573    2.509709  170710.984450   
min              1974-07-01 00:00:00  1974.000000    1.000000    8358.543702   
25%              1987-04-23 18:00:00  1987.000000    2.000000   57467.695611   
50%              2000-02-15 12:00:00  2000.000000    3.000000  158853.000000   
75%              2012-12-09 00:00:00  2012.750000    3.750000  270025.500000   
max              2025-10-01 00:00:00  2025.000000    4.000000  450529.000000   
std                              NaN    14.904390    1.120715  118259.030898   

       demanda_nacional  consumo_privado_nacional  consumo_privado_interior  \
count        206.000000                206.000000                206.000000   
mean      170165.857039              99721.111045             103162.586020   


In [25]:
for nombre, df in [('DEMANDA CORRIENTES', df_dem_corrientes),
                    ('OFERTA CORRIENTES', df_of_corrientes),
                    ('DEMANDA CONSTANTES 2025', df_dem_constantes),
                    ('OFERTA CONSTANTES 2025', df_of_constantes)]:
    print(f'Información acerca de {nombre}:')
    print(f'{df.info()}')

Información acerca de DEMANDA CORRIENTES:
<class 'pandas.core.frame.DataFrame'>
Index: 206 entries, 18 to 223
Data columns (total 16 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   fecha                         206 non-null    datetime64[ns]
 1   año                           206 non-null    int64         
 2   trimestre                     206 non-null    int64         
 3   pib_pm                        206 non-null    float64       
 4   demanda_nacional              206 non-null    float64       
 5   consumo_privado_nacional      206 non-null    float64       
 6   consumo_privado_interior      206 non-null    float64       
 7   consumo_publico               206 non-null    float64       
 8   fbcf                          206 non-null    float64       
 9   variacion_existencias         206 non-null    float64       
 10  exportacion_bienes_servicios  206 non-null    float64       